# analyzer

In [4]:
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List
from collections import Counter



class ClimateAnalysis:
    """Analyze and evaluate processed climate reports"""

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.reports = []

    def load_all_reports(self) -> List[Dict]:
        """Load all BERT analysis results from cache"""
        bert_files = list(self.cache_dir.glob("*_bert.json"))

        if not bert_files:
            print(f"❌ No bert.json files found in {self.cache_dir}")
            return []

        print(f"📂 Found {len(bert_files)} analyzed reports")

        self.reports = []
        for file in bert_files:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Add filename for reference
                data['filename'] = file.stem.replace('_bert', '')
                self.reports.append(data)

        print(f"✓ Loaded {len(self.reports)} reports successfully")
        return self.reports

    def create_summary_df(self) -> pd.DataFrame:
        """Create summary DataFrame with key metrics per report"""
        if not self.reports:
            self.load_all_reports()

        summaries = []
        for report in self.reports:
            # Count chunks by category
            chunks = report.get('chunks', [])

            # Specificity counts
            spec_counts = Counter(c.get('specificity_label', 'unknown') for c in chunks)

            # Sentiment counts
            sent_counts = Counter(c.get('sentiment_label', 'unknown') for c in chunks)

            # Commitment counts
            commit_counts = Counter(c.get('commitment_label', 'unknown') for c in chunks)

            # Average scores
            spec_scores = [c.get('specificity_score', 0) for c in chunks if 'specificity_score' in c]
            sent_scores = [c.get('sentiment_score', 0) for c in chunks if 'sentiment_score' in c]
            commit_scores = [c.get('commitment_score', 0) for c in chunks if 'commitment_score' in c]

            summary = {
                'filename': report['filename'],
                'language': report.get('language', 'unknown'),
                'total_chunks': report.get('total_chunks', 0),
                'climate_chunks': report.get('climate_chunks', 0),
                'kept_percentage': report.get('kept_percentage', 0),

                # Specificity
                'spec_specific': spec_counts.get('spec', 0),
                'spec_non_specific': spec_counts.get('non', 0),
                'spec_avg_score': sum(spec_scores) / len(spec_scores) if spec_scores else 0,

                # Sentiment
                'sent_opportunity': sent_counts.get('opportunity', 0),
                'sent_neutral': sent_counts.get('neutral', 0),
                'sent_risk': sent_counts.get('risk', 0),
                'sent_avg_score': sum(sent_scores) / len(sent_scores) if sent_scores else 0,

                # Commitment
                'commit_yes': commit_counts.get('yes', 0),
                'commit_no': commit_counts.get('no', 0),
                'commit_avg_score': sum(commit_scores) / len(commit_scores) if commit_scores else 0,
            }

            summaries.append(summary)

        df = pd.DataFrame(summaries)

        # Add percentage columns
        df['spec_specific_pct'] = (df['spec_specific'] / df['climate_chunks'] * 100).round(1)
        df['sent_opportunity_pct'] = (df['sent_opportunity'] / df['climate_chunks'] * 100).round(1)
        df['commit_yes_pct'] = (df['commit_yes'] / df['climate_chunks'] * 100).round(1)

        return df

    def print_overall_stats(self):
        """Print overall statistics across all reports"""
        if not self.reports:
            self.load_all_reports()

        df = self.create_summary_df()

        print(f"\n{'='*80}")
        print(f"OVERALL STATISTICS - {len(self.reports)} REPORTS")
        print(f"{'='*80}\n")

        print(f"📊 Total Chunks Analysis:")
        print(f"  • Total chunks extracted: {df['total_chunks'].sum():,}")
        print(f"  • Climate-related chunks: {df['climate_chunks'].sum():,}")
        print(f"  • Average kept %: {df['kept_percentage'].mean():.1f}%")

        print(f"\n🎯 Specificity Distribution:")
        print(f"  • Specific: {df['spec_specific'].sum():,} ({df['spec_specific_pct'].mean():.1f}% avg)")
        print(f"  • Non-specific: {df['spec_non_specific'].sum():,}")

        print(f"\n💭 Sentiment Distribution:")
        print(f"  • Opportunity: {df['sent_opportunity'].sum():,} ({df['sent_opportunity_pct'].mean():.1f}% avg)")
        print(f"  • Neutral: {df['sent_neutral'].sum():,}")
        print(f"  • Risk: {df['sent_risk'].sum():,}")

        print(f"\n✅ Commitment Distribution:")
        print(f"  • Committed: {df['commit_yes'].sum():,} ({df['commit_yes_pct'].mean():.1f}% avg)")
        print(f"  • Not committed: {df['commit_no'].sum():,}")

        print(f"\n🌍 Language Distribution:")
        for lang, count in df['language'].value_counts().items():
            print(f"  • {lang}: {count} reports")

        print(f"\n{'='*80}\n")

        return df

    def get_top_reports(self, metric='commit_yes_pct', n=10):
        """Get top N reports by specified metric"""
        df = self.create_summary_df()

        if metric not in df.columns:
            print(f"❌ Unknown metric: {metric}")
            return None

        top = df.nlargest(n, metric)[['filename', 'climate_chunks', metric]]

        print(f"\n{'='*80}")
        print(f"TOP {n} REPORTS BY {metric.upper()}")
        print(f"{'='*80}\n")
        print(top.to_string(index=False))
        print(f"\n{'='*80}\n")

        return top

    def analyze_report(self, filename: str):
        """Deep dive analysis of a single report"""
        report = next((r for r in self.reports if r['filename'] == filename), None)

        if not report:
            print(f"❌ Report not found: {filename}")
            return None

        chunks = report.get('chunks', [])

        print(f"\n{'='*80}")
        print(f"DETAILED ANALYSIS: {filename}")
        print(f"{'='*80}\n")

        print(f"📄 Basic Info:")
        print(f"  • Language: {report.get('language', 'unknown')}")
        print(f"  • Total chunks: {report.get('total_chunks', 0):,}")
        print(f"  • Climate chunks: {len(chunks):,} ({report.get('kept_percentage', 0):.1f}%)")

        # Specificity breakdown
        spec_dist = Counter(c.get('specificity_label') for c in chunks)
        print(f"\n🎯 Specificity:")
        for label, count in spec_dist.most_common():
            pct = count / len(chunks) * 100
            print(f"  • {label}: {count} ({pct:.1f}%)")

        # Sentiment breakdown
        sent_dist = Counter(c.get('sentiment_label') for c in chunks)
        print(f"\n💭 Sentiment:")
        for label, count in sent_dist.most_common():
            pct = count / len(chunks) * 100
            print(f"  • {label}: {count} ({pct:.1f}%)")

        # Commitment breakdown
        commit_dist = Counter(c.get('commitment_label') for c in chunks)
        print(f"\n✅ Commitment:")
        for label, count in commit_dist.most_common():
            pct = count / len(chunks) * 100
            print(f"  • {label}: {count} ({pct:.1f}%)")

        # Show example chunks
        print(f"\n📝 Example Chunks:")

        # Most committed + specific + opportunity
        best_chunks = sorted(
            chunks,
            key=lambda c: (
                c.get('commitment_label') == 'yes',
                c.get('specificity_label') == 'spec',
                c.get('sentiment_label') == 'opportunity',
                c.get('commitment_score', 0)
            ),
            reverse=True
        )[:3]

        for i, chunk in enumerate(best_chunks, 1):
            text = chunk['text'][:200] + '...' if len(chunk['text']) > 200 else chunk['text']
            print(f"\n  Example {i}:")
            print(f"    Specificity: {chunk.get('specificity_label')} ({chunk.get('specificity_score', 0):.2f})")
            print(f"    Sentiment: {chunk.get('sentiment_label')} ({chunk.get('sentiment_score', 0):.2f})")
            print(f"    Commitment: {chunk.get('commitment_label')} ({chunk.get('commitment_score', 0):.2f})")
            print(f"    Text: {text}")

        print(f"\n{'='*80}\n")

        return report

    def export_summary(self, output_file="climate_analysis_summary.csv"):
        """Export summary to CSV"""
        df = self.create_summary_df()
        df.to_csv(output_file, index=False)
        print(f"✓ Exported summary to {output_file}")
        return df

    def compare_companies(self, company_keywords: Dict[str, List[str]]):
        """Compare companies based on filename patterns

        Example:
            company_keywords = {
                'Baowu': ['baowu', 'baoshan'],
                'ArcelorMittal': ['arcelor', 'mittal'],
                'POSCO': ['posco']
            }
        """
        df = self.create_summary_df()

        # Assign companies
        def get_company(filename):
            filename_lower = filename.lower()
            for company, keywords in company_keywords.items():
                if any(kw in filename_lower for kw in keywords):
                    return company
            return 'Other'

        df['company'] = df['filename'].apply(get_company)

        # Group by company
        company_stats = df.groupby('company').agg({
            'climate_chunks': 'sum',
            'kept_percentage': 'mean',
            'spec_specific_pct': 'mean',
            'sent_opportunity_pct': 'mean',
            'commit_yes_pct': 'mean',
            'filename': 'count'
        }).rename(columns={'filename': 'num_reports'})

        print(f"\n{'='*80}")
        print(f"COMPANY COMPARISON")
        print(f"{'='*80}\n")
        print(company_stats.round(1).to_string())
        print(f"\n{'='*80}\n")

        return company_stats

    def find_best_practices(self, min_commitment_score=0.95, min_specificity_score=0.8):
        """Find chunks with strong commitments and high specificity"""
        if not self.reports:
            self.load_all_reports()

        best_practices = []

        for report in self.reports:
            for chunk in report.get('chunks', []):
                if (chunk.get('commitment_label') == 'yes' and
                    chunk.get('commitment_score', 0) >= min_commitment_score and
                    chunk.get('specificity_label') == 'spec' and
                    chunk.get('specificity_score', 0) >= min_specificity_score):

                    best_practices.append({
                        'filename': report['filename'],
                        'text': chunk['text'][:300],
                        'commitment_score': chunk['commitment_score'],
                        'specificity_score': chunk['specificity_score'],
                        'sentiment': chunk.get('sentiment_label')
                    })

        print(f"\n{'='*80}")
        print(f"BEST PRACTICES - Strong Commitments ({len(best_practices)} found)")
        print(f"{'='*80}\n")

        # Sort by commitment score
        best_practices.sort(key=lambda x: x['commitment_score'], reverse=True)

        for i, practice in enumerate(best_practices[:10], 1):
            print(f"{i}. {practice['filename']}")
            print(f"   Commitment: {practice['commitment_score']:.3f} | "
                  f"Specificity: {practice['specificity_score']:.3f} | "
                  f"Sentiment: {practice['sentiment']}")
            print(f"   {practice['text']}...")
            print()

        return best_practices




# step-by-step

In [2]:
# Initialize
analyzer = ClimateAnalysis(cache_dir="cache")

# Load all reports
analyzer.load_all_reports()

# Overall statistics
df = analyzer.print_overall_stats()

# Top committed reports
analyzer.get_top_reports('commit_yes_pct', n=10)

# Analyze specific report
analyzer.analyze_report('015_2021_green_report')

# Find best practices
best = analyzer.find_best_practices(min_commitment_score=0.95)

# Export summary
analyzer.export_summary("climate_summary.csv")

# Compare companies (customize keywords)
companies = {
    'Baowu': ['baowu'],
    'ArcelorMittal': ['arcelor'],
    'POSCO': ['posco']
}
analyzer.compare_companies(companies)

📂 Found 170 analyzed reports
✓ Loaded 170 reports successfully

OVERALL STATISTICS - 170 REPORTS

📊 Total Chunks Analysis:
  • Total chunks extracted: 35,300
  • Climate-related chunks: 15,320
  • Average kept %: 52.3%

🎯 Specificity Distribution:
  • Specific: 9,295 (61.8% avg)
  • Non-specific: 6,025

💭 Sentiment Distribution:
  • Opportunity: 5,625 (39.1% avg)
  • Neutral: 7,061
  • Risk: 2,634

✅ Commitment Distribution:
  • Committed: 9,918 (63.2% avg)
  • Not committed: 5,402

🌍 Language Distribution:
  • en: 141 reports
  • de: 18 reports
  • zh: 5 reports
  • es: 3 reports
  • it: 2 reports
  • et: 1 reports



TOP 10 REPORTS BY COMMIT_YES_PCT

                                filename  climate_chunks  commit_yes_pct
     daten_und_fakten_dillinger_2020_eng               1           100.0
015_2018_corporate_responsibility_report              64            98.4
015_2024_corporate_responsibility_report             123            97.6
015_2020_corporate_responsibility_report       

,climate_chunks,kept_percentage,spec_specific_pct,sent_opportunity_pct,commit_yes_pct,num_reports
company,,,,,,
Other,15320,52.348348,61.828402,39.127811,63.152663,170


# dir - whole pipe

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional
from collections import Counter, defaultdict
import re
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns


class ClimateReportAnalyzer2:
    """Multi-scope analysis: chunk → PDF → company → sector"""

    def __init__(self, cache_dir="cache"):
        self.cache_dir = Path(cache_dir)
        self.raw_reports = []

        # Analysis levels
        self.chunk_df = None      # Level 1: Individual chunks
        self.pdf_df = None        # Level 2: Per-report aggregates
        self.company_df = None    # Level 3: Time series per company
        self.sector_df = None     # Level 4: Cross-company

    # ========================================================================
    # LEVEL 0: Load & Parse
    # ========================================================================

    def load_all_reports(self) -> List[Dict]:
        """Load all BERT analysis results"""
        bert_files = list(self.cache_dir.glob("*_bert.json"))

        if not bert_files:
            print(f"❌ No *_bert.json files found in {self.cache_dir}")
            return []

        print(f"📂 Found {len(bert_files)} analyzed reports")

        self.raw_reports = []
        for file in bert_files:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                data['_cache_filename'] = file.stem
                self.raw_reports.append(data)

        print(f"✓ Loaded {len(self.raw_reports)} reports")
        return self.raw_reports

    def _parse_filename(self, cache_filename: str) -> Dict:
        """Extract company, year, report_type from filename

        Expected format: {company}_{year}_{type}_bert
        e.g., '015_2021_green_report_bert' or 'thyssenkrupp_2020_annual_bert'

        Returns: {'company': str, 'year': int, 'report_type': str}
        """
        # Remove '_bert' suffix
        name = cache_filename.replace('_bert', '')

        # Try to extract year (4 consecutive digits)
        year_match = re.search(r'(\d{4})', name)
        year = int(year_match.group(1)) if year_match else None

        # Try to extract report type (sustainability, annual, green, etc.)
        type_keywords = ['sustainability', 'annual', 'green', 'esg', 'csrreport']
        report_type = None
        for keyword in type_keywords:
            if keyword in name.lower():
                report_type = keyword
                break

        # Company name is the rest (before year)
        if year_match:
            company = name[:year_match.start()].strip('_').lower()
        else:
            company = name.lower()

        return {
            'company': company,
            'year': year,
            'report_type': report_type or 'unknown'
        }

    # ========================================================================
    # LEVEL 1: Chunk Analysis
    # ========================================================================

    def build_chunk_df(self) -> pd.DataFrame:
        """Build chunk-level DataFrame with all metadata

        Columns:
        - chunk_id, pdf_id, company, year, report_type
        - text, text_len_chars, text_len_tokens (approx)
        - climate_score, climate_label
        - specificity_label, specificity_score, specificity_confidence
        - sentiment_label, sentiment_score, sentiment_confidence
        - commitment_label, commitment_score, commitment_confidence
        - source (translated/original), language
        """
        if not self.raw_reports:
            self.load_all_reports()

        rows = []
        pdf_id = 0
        chunk_id = 0

        for report in self.raw_reports:
            pdf_id += 1
            metadata = self._parse_filename(report['_cache_filename'])

            # PDF-level info
            pdf_info = {
                'pdf_id': pdf_id,
                'cache_filename': report['_cache_filename'],
                'company': metadata['company'],
                'year': metadata['year'],
                'report_type': metadata['report_type'],
                'language': report.get('language', 'unknown'),
                'source': report.get('source', 'unknown'),  # translated/original
                'total_chunks': report.get('total_chunks', 0),
                'climate_chunks': report.get('climate_chunks', 0),
                'kept_percentage': report.get('kept_percentage', 0)
            }

            # Chunk-level info
            for chunk in report.get('chunks', []):
                chunk_id += 1

                text = chunk.get('text', '')

                row = {
                    'chunk_id': chunk_id,
                    'text': text,
                    'text_len_chars': len(text),
                    'text_len_tokens': len(text) // 4,  # Rough estimate

                    # Detection
                    'detector_label': chunk.get('detector_label', 'unknown'),
                    'detector_score': chunk.get('detector_score', 0),

                    # Specificity
                    'specificity_label': chunk.get('specificity_label', 'unknown'),
                    'specificity_score': chunk.get('specificity_score', 0),

                    # Sentiment
                    'sentiment_label': chunk.get('sentiment_label', 'unknown'),
                    'sentiment_score': chunk.get('sentiment_score', 0),

                    # Commitment
                    'commitment_label': chunk.get('commitment_label', 'unknown'),
                    'commitment_score': chunk.get('commitment_score', 0),

                    **pdf_info
                }
                rows.append(row)

        self.chunk_df = pd.DataFrame(rows)

        # Add confidence bins
        for metric in ['specificity', 'sentiment', 'commitment']:
            score_col = f'{metric}_score'
            self.chunk_df[f'{metric}_confidence_bin'] = pd.cut(
                self.chunk_df[score_col],
                bins=[0, 0.6, 0.8, 1.0],
                labels=['low', 'medium', 'high']
            )

        print(f"✓ Built chunk_df: {len(self.chunk_df):,} chunks from {pdf_id} PDFs")
        return self.chunk_df

    # ========================================================================
    # LEVEL 2: PDF Analysis (Per Report)
    # ========================================================================

    def build_pdf_df(self) -> pd.DataFrame:
        """Aggregate chunks to PDF-level metrics

        Uses weighted averages: confidence × length

        Metrics:
        - climate_fraction: % of chunks that are climate-related
        - specificity_mean: weighted avg of specificity scores
        - commitment_fraction: % labeled 'yes' (weighted by confidence)
        - sentiment distribution: % opportunity/neutral/risk
        - total_tokens: sum of all chunk tokens
        """
        if self.chunk_df is None:
            self.build_chunk_df()

        df = self.chunk_df.copy()

        # Group by PDF
        pdf_groups = df.groupby('pdf_id')

        rows = []
        for pdf_id, group in pdf_groups:
            # Basic info (same for all chunks in group)
            first = group.iloc[0]

            # Calculate weighted metrics
            weights = group['text_len_tokens'] * group['detector_score']
            total_weight = weights.sum()

            # Specificity: weighted mean
            spec_mean = (
                (group['specificity_score'] * weights).sum() / total_weight
                if total_weight > 0 else 0
            )

            # Commitment: soft count (confidence-weighted fraction)
            commit_yes = group[group['commitment_label'] == 'yes']
            commit_fraction = (
                (commit_yes['commitment_score'] * commit_yes['text_len_tokens']).sum() /
                group['text_len_tokens'].sum()
                if len(group) > 0 else 0
            )

            # Sentiment distribution
            sent_dist = group['sentiment_label'].value_counts(normalize=True)

            row = {
                'pdf_id': pdf_id,
                'cache_filename': first['cache_filename'],
                'company': first['company'],
                'year': first['year'],
                'report_type': first['report_type'],
                'language': first['language'],
                'source': first['source'],

                # Chunk counts
                'total_chunks': first['total_chunks'],
                'climate_chunks': len(group),
                'climate_fraction': first['kept_percentage'] / 100,

                # Tokens
                'total_tokens': group['text_len_tokens'].sum(),

                # Specificity
                'specificity_mean': spec_mean,
                'specificity_spec_frac': (group['specificity_label'] == 'spec').mean(),

                # Commitment
                'commitment_fraction': commit_fraction,
                'commitment_yes_count': len(commit_yes),

                # Sentiment
                'sentiment_opportunity_frac': sent_dist.get('opportunity', 0),
                'sentiment_neutral_frac': sent_dist.get('neutral', 0),
                'sentiment_risk_frac': sent_dist.get('risk', 0),

                # Confidence distribution
                'specificity_high_conf_frac': (group['specificity_confidence_bin'] == 'high').mean(),
                'commitment_high_conf_frac': (group['commitment_confidence_bin'] == 'high').mean(),
            }

            rows.append(row)

        self.pdf_df = pd.DataFrame(rows)

        # Sort by company, year
        self.pdf_df = self.pdf_df.sort_values(['company', 'year'])

        print(f"✓ Built pdf_df: {len(self.pdf_df)} reports")
        return self.pdf_df

    # ========================================================================
    # LEVEL 3: Company Analysis (Time Series)
    # ========================================================================

    def build_company_df(self) -> pd.DataFrame:
        """Time series analysis per company

        Metrics per company:
        - report_availability: years covered, gaps
        - trend metrics: Δ%, slope, R², p-value
        - volatility: std of year-over-year changes
        - paris_alignment: compare to required -4.2%/yr (1.5°C) or -2.5%/yr (2°C)
        """
        if self.pdf_df is None:
            self.build_pdf_df()

        company_groups = self.pdf_df.groupby('company')

        rows = []
        for company, group in company_groups:
            group = group.sort_values('year')

            # Report availability
            years = group['year'].dropna().astype(int).tolist()
            year_range = (min(years), max(years)) if years else (None, None)
            n_years = len(years)
            expected_years = year_range[1] - year_range[0] + 1 if year_range[0] else 0
            gaps = expected_years - n_years

            # Get time series
            ts = group.set_index('year').sort_index()

            # Calculate trends for key metrics
            metrics_to_track = ['commitment_fraction', 'specificity_mean', 'climate_fraction']

            trends = {}
            for metric in metrics_to_track:
                if len(ts) >= 2 and metric in ts.columns:
                    x = np.arange(len(ts))
                    y = ts[metric].values

                    # Linear regression
                    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

                    # Year-over-year changes
                    yoy_changes = ts[metric].pct_change().dropna()

                    trends[metric] = {
                        'start': y[0] if len(y) > 0 else None,
                        'end': y[-1] if len(y) > 0 else None,
                        'total_change_pct': ((y[-1] - y[0]) / y[0] * 100) if len(y) > 0 and y[0] != 0 else None,
                        'avg_annual_change_pct': (slope / y.mean() * 100) if len(y) > 0 and y.mean() != 0 else None,
                        'slope': slope,
                        'r_squared': r_value**2,
                        'p_value': p_value,
                        'volatility': yoy_changes.std() if len(yoy_changes) > 0 else None
                    }
                else:
                    trends[metric] = {}

            row = {
                'company': company,
                'n_reports': len(group),
                'year_start': year_range[0],
                'year_end': year_range[1],
                'years_covered': n_years,
                'expected_years': expected_years,
                'missing_years': gaps,
                'report_availability': n_years / expected_years if expected_years > 0 else 0,

                # Languages used
                'languages': ','.join(group['language'].unique()),
                'translated_reports': (group['source'] == 'translated').sum(),

                # Aggregate metrics (latest year)
                'latest_climate_fraction': ts['climate_fraction'].iloc[-1] if len(ts) > 0 else None,
                'latest_commitment_fraction': ts['commitment_fraction'].iloc[-1] if len(ts) > 0 else None,
                'latest_specificity_mean': ts['specificity_mean'].iloc[-1] if len(ts) > 0 else None,
            }

            # Add trend metrics
            for metric in metrics_to_track:
                for key, val in trends[metric].items():
                    row[f'{metric}_{key}'] = val

            rows.append(row)

        self.company_df = pd.DataFrame(rows)

        # Paris alignment (for commitment_fraction)
        self.company_df['paris_1.5C_gap'] = self.company_df['commitment_fraction_avg_annual_change_pct'] - (-4.2)
        self.company_df['paris_2C_gap'] = self.company_df['commitment_fraction_avg_annual_change_pct'] - (-2.5)

        print(f"✓ Built company_df: {len(self.company_df)} companies")
        return self.company_df

    # ========================================================================
    # LEVEL 4: Sector Analysis
    # ========================================================================

    def build_sector_df(self) -> pd.DataFrame:
        """Cross-company sector statistics"""
        if self.pdf_df is None:
            self.build_pdf_df()

        # Overall statistics
        stats = {
            'total_companies': self.pdf_df['company'].nunique(),
            'total_reports': len(self.pdf_df),
            'total_chunks': self.chunk_df['chunk_id'].nunique() if self.chunk_df is not None else 0,
            'total_climate_chunks': self.pdf_df['climate_chunks'].sum(),

            # Averages
            'avg_climate_fraction': self.pdf_df['climate_fraction'].mean(),
            'avg_commitment_fraction': self.pdf_df['commitment_fraction'].mean(),
            'avg_specificity_mean': self.pdf_df['specificity_mean'].mean(),

            # Distributions
            'median_climate_fraction': self.pdf_df['climate_fraction'].median(),
            'std_climate_fraction': self.pdf_df['climate_fraction'].std(),

            # Language/translation
            'translated_reports': (self.pdf_df['source'] == 'translated').sum(),
            'translated_pct': (self.pdf_df['source'] == 'translated').mean() * 100,
        }

        self.sector_df = pd.DataFrame([stats])

        print(f"✓ Built sector_df: sector-wide statistics")
        return self.sector_df

    # ========================================================================
    # Build All Levels
    # ========================================================================

    def build_all(self):
        """Build all analysis levels"""
        print(f"\n{'='*80}")
        print(f"BUILDING MULTI-LEVEL ANALYSIS")
        print(f"{'='*80}\n")

        self.load_all_reports()
        self.build_chunk_df()
        self.build_pdf_df()
        self.company_df()
        self.build_sector_df()

        print(f"\n{'='*80}")
        print(f"✓ All levels built successfully")
        print(f"{'='*80}\n")

    # ========================================================================
    # Export
    # ========================================================================

    def export_all(self, output_dir="analysis_output"):
        """Export all DataFrames to CSV"""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        if self.chunk_df is not None:
            self.chunk_df.to_csv(output_path / "1_chunks.csv", index=False)
            print(f"✓ Exported chunks → {output_path}/1_chunks.csv")

        if self.pdf_df is not None:
            self.pdf_df.to_csv(output_path / "2_pdfs.csv", index=False)
            print(f"✓ Exported PDFs → {output_path}/2_pdfs.csv")

        if self.company_df is not None:
            self.company_df.to_csv(output_path / "3_companies.csv", index=False)
            print(f"✓ Exported companies → {output_path}/3_companies.csv")

        if self.sector_df is not None:
            self.sector_df.to_csv(output_path / "4_sector.csv", index=False)
            print(f"✓ Exported sector → {output_path}/4_sector.csv")

    # ========================================================================
    # Quick Stats
    # ========================================================================

    def print_summary(self):
        """Print summary statistics"""
        if self.sector_df is None:
            self.build_all()

        print(f"\n{'='*80}")
        print(f"SECTOR SUMMARY")
        print(f"{'='*80}\n")

        print(f"📊 Scope:")
        print(f"  • Companies: {self.sector_df['total_companies'].iloc[0]}")
        print(f"  • Reports: {self.sector_df['total_reports'].iloc[0]}")
        print(f"  • Total chunks: {self.sector_df['total_chunks'].iloc[0]:,}")
        print(f"  • Climate chunks: {self.sector_df['total_climate_chunks'].iloc[0]:,}")

        print(f"\n📈 Averages:")
        print(f"  • Climate fraction: {self.sector_df['avg_climate_fraction'].iloc[0]:.1%}")
        print(f"  • Commitment fraction: {self.sector_df['avg_commitment_fraction'].iloc[0]:.1%}")
        print(f"  • Specificity mean: {self.sector_df['avg_specificity_mean'].iloc[0]:.3f}")

        print(f"\n🌍 Languages:")
        print(f"  • Translated reports: {self.sector_df['translated_reports'].iloc[0]} ({self.sector_df['translated_pct'].iloc[0]:.1f}%)")

        print(f"\n{'='*80}\n")


# ==============================================================================
# USAGE
# ==============================================================================

if __name__ == "__main__":
    analyzer = ClimateReportAnalyzer2(cache_dir="cache")

    # Build all levels
    analyzer.build_all()

    # Print summary
    analyzer.print_summary()

    # Export to CSV
    analyzer.export_all("analysis_output")

    # Access DataFrames for further analysis
    chunks = analyzer.chunk_df
    pdfs = analyzer.pdf_df
    companies = analyzer.company_df
    sector = analyzer.sector_df


BUILDING MULTI-LEVEL ANALYSIS

📂 Found 170 analyzed reports
✓ Loaded 170 reports
✓ Built chunk_df: 15,320 chunks from 170 PDFs
✓ Built pdf_df: 169 reports


TypeError: 'NoneType' object is not callable

In [ ]:
"""
ClimateReportAnalyzer - Optimized with Helsinki-NLP Translation

Key improvements:
1. Helsinki-NLP models for de/es/it/zh → en (2-3x faster than NLLB)
2. Falls back to NLLB for unsupported languages
3. torch.compile() for additional speedup
4. Optimized memory management for 4GB VRAM
"""

import spacy
import langid
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer, pipeline
import fitz  # PyMuPDF
import torch
import json
import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
import time
from collections import Counter
import gc
import os

# === CRITICAL: Must be set BEFORE importing torch ===
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.7,max_split_size_mb:256"


_nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer", "tagger"])
_nlp.max_length = 2_000_000


class ClimateReportAnalyzer:
    # Chunking settings
    MIN_CHARS = 600
    MAX_CHARS = 1600
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 32   # For BERT models

    # Translation settings - OPTIMIZED FOR HELSINKI + 4GB VRAM
    TRANSLATE_BATCH_SIZE = 32      # Helsinki can handle larger batches!
    TRANSLATE_INPUT_MAX_TOKENS = 512  # Helsinki handles 512 well
    TRANSLATE_OUTPUT_MAX_TOKENS = 512

    # Helsinki-NLP models (fast, language-specific)
    HELSINKI_MODELS = {
        'de': 'Helsinki-NLP/opus-mt-de-en',
        'es': 'Helsinki-NLP/opus-mt-es-en',
        'it': 'Helsinki-NLP/opus-mt-it-en',
        'zh': 'Helsinki-NLP/opus-mt-zh-en',
        'fr': 'Helsinki-NLP/opus-mt-fr-en',
        'nl': 'Helsinki-NLP/opus-mt-nl-en',
        'pt': 'Helsinki-NLP/opus-mt-ROMANCE-en',  # Portuguese via Romance
        'pl': 'Helsinki-NLP/opus-mt-pl-en',
        'ru': 'Helsinki-NLP/opus-mt-ru-en',
        'ja': 'Helsinki-NLP/opus-mt-ja-en',
        'ko': 'Helsinki-NLP/opus-mt-ko-en',
    }

    # NLLB fallback for languages without Helsinki model
    NLLB_LANG_MAP = {
        'de': 'deu_Latn', 'fr': 'fra_Latn', 'es': 'spa_Latn',
        'it': 'ita_Latn', 'pt': 'por_Latn', 'nl': 'nld_Latn',
        'pl': 'pol_Latn', 'sv': 'swe_Latn', 'fi': 'fin_Latn',
        'no': 'nob_Latn', 'nn': 'nno_Latn', 'da': 'dan_Latn',
        'ru': 'rus_Cyrl', 'uk': 'ukr_Cyrl', 'zh': 'zho_Hans',
        'ja': 'jpn_Jpan', 'ko': 'kor_Hang', 'ar': 'arb_Arab',
        'tr': 'tur_Latn', 'cs': 'ces_Latn', 'ro': 'ron_Latn',
        'hu': 'hun_Latn', 'en': 'eng_Latn',
    }

    def __init__(self, cache_dir="cache", use_helsinki=True, use_torch_compile=True):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.pdf_path = None

        self.use_helsinki = use_helsinki
        self.use_torch_compile = use_torch_compile

        self.device = 0 if torch.cuda.is_available() else -1
        self.models = {}
        self.translator = None
        self.translator_lang = None  # Track which language model is loaded

        # In-memory cache
        self.prep_data = None
        self.bert_data = None

        # Report GPU status
        if self.device >= 0:
            gpu_name = torch.cuda.get_device_name(0)
            total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"✓ GPU: {gpu_name} ({total_mem:.1f}GB)")
            print(
                f"✓ PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'NOT SET')}")
            print(
                f"✓ Translation backend: {'Helsinki-NLP (fast)' if use_helsinki else 'NLLB (universal)'}")
            self._clear_gpu_memory()

    def _get_gpu_memory_info(self) -> Dict:
        """Get current GPU memory status."""
        if self.device < 0:
            return {'available': False}
        return {
            'available': True,
            'allocated_gb': torch.cuda.memory_allocated(0) / 1e9,
            'reserved_gb': torch.cuda.memory_reserved(0) / 1e9,
            'total_gb': torch.cuda.get_device_properties(0).total_memory / 1e9,
            'free_gb': (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
        }

    def _clear_gpu_memory(self):
        """Standard GPU memory cleanup."""
        if self.device >= 0:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    def _emergency_gpu_cleanup(self):
        """Aggressive GPU cleanup for OOM recovery."""
        if self.device < 0:
            return

        before = self._get_gpu_memory_info()
        print(
            f"  Emergency cleanup: {before['allocated_gb']:.2f}GB allocated before")

        # Unload all models
        if self.translator:
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except:
                pass
            self.translator = None
            self.translator_lang = None

        for name in list(self.models.keys()):
            try:
                self.models[name].model.cpu()
                del self.models[name]
            except:
                pass
        self.models = {}

        gc.collect()

        # Move any remaining CUDA tensors to CPU
        for obj in gc.get_objects():
            try:
                if torch.is_tensor(obj) and obj.is_cuda:
                    obj.data = obj.data.cpu()
            except:
                pass

        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        gc.collect()

        after = self._get_gpu_memory_info()
        print(
            f"  Emergency cleanup: {after['allocated_gb']:.2f}GB allocated after")

    def _unload_translator(self):
        """Unload translation model to free memory."""
        if self.translator:
            print("  Unloading translation model...")
            try:
                if 'model' in self.translator:
                    self.translator['model'].cpu()
                    del self.translator['model']
                if 'tokenizer' in self.translator:
                    del self.translator['tokenizer']
            except Exception as e:
                print(f"  ⚠ Error during unload: {e}")

            self.translator = None
            self.translator_lang = None
            self._clear_gpu_memory()

            info = self._get_gpu_memory_info()
            print(f"  GPU memory after unload: {info['allocated_gb']:.2f}GB")

    # ==================== HELSINKI-NLP TRANSLATION ====================

    def _load_helsinki_translator(self, src_lang: str):
        """Load Helsinki-NLP model for specific language."""

        # Check if correct model already loaded
        if self.translator and self.translator_lang == src_lang:
            return self.translator

        # Unload previous model if different language
        if self.translator:
            self._unload_translator()

        model_name = self.HELSINKI_MODELS.get(src_lang)
        if not model_name:
            return None

        print(f"Loading Helsinki-NLP model: {model_name}")
        start = time.time()

        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device >= 0 else torch.float32,
        )

        if self.device >= 0:
            model = model.cuda()

            # Optional: torch.compile for extra speed (PyTorch 2.0+)
            if self.use_torch_compile and hasattr(torch, 'compile'):
                try:
                    print("  Applying torch.compile()...")
                    model = torch.compile(model, mode="reduce-overhead")
                except Exception as e:
                    print(f"  torch.compile failed: {e}")

        model.eval()

        load_time = time.time() - start
        info = self._get_gpu_memory_info()
        print(
            f"✓ Helsinki model loaded in {load_time:.1f}s ({info['allocated_gb']:.2f}GB)")

        self.translator = {'model': model,
                           'tokenizer': tokenizer, 'type': 'helsinki'}
        self.translator_lang = src_lang
        return self.translator

    def _load_nllb_translator(self):
        """Load NLLB model as fallback."""
        if self.translator and self.translator.get('type') == 'nllb':
            return self.translator

        if self.translator:
            self._unload_translator()

        print("Loading NLLB-200 translation model (fallback)...")
        model_name = "facebook/nllb-200-distilled-600M"

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if self.device >= 0 else torch.float32,
            low_cpu_mem_usage=True,
        )

        if self.device >= 0:
            model = model.cuda()

        model.eval()

        info = self._get_gpu_memory_info()
        print(f"✓ NLLB model loaded ({info['allocated_gb']:.2f}GB)")

        self.translator = {'model': model,
                           'tokenizer': tokenizer, 'type': 'nllb'}
        self.translator_lang = 'nllb'
        return self.translator

    def _translate_chunks_helsinki(self, chunks: List[str], src_lang: str) -> List[str]:
        """Translate using Helsinki-NLP (fast, language-specific)."""
        translator = self._load_helsinki_translator(src_lang)
        model = translator['model']
        tokenizer = translator['tokenizer']

        print(f"Translating {src_lang} → en (Helsinki-NLP)")
        print(
            f"Batch size: {self.TRANSLATE_BATCH_SIZE}, Max tokens: {self.TRANSLATE_INPUT_MAX_TOKENS}")

        translated = []

        with torch.no_grad():
            for batch_idx, i in enumerate(tqdm(range(0, len(chunks), self.TRANSLATE_BATCH_SIZE), desc="Translating")):
                batch = chunks[i:i + self.TRANSLATE_BATCH_SIZE]

                try:
                    inputs = tokenizer(
                        batch,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=self.TRANSLATE_INPUT_MAX_TOKENS
                    )

                    if self.device >= 0:
                        inputs = {k: v.cuda() for k, v in inputs.items()}

                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                        num_beams=1,      # Greedy for speed
                        do_sample=False,
                        use_cache=True,
                    )

                    batch_translations = tokenizer.batch_decode(
                        outputs, skip_special_tokens=True)
                    translated.extend(batch_translations)

                    del inputs, outputs

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print(
                            f"\n⚠ OOM at batch {batch_idx}, processing one by one...")
                        torch.cuda.empty_cache()

                        for chunk in batch:
                            try:
                                inputs = tokenizer([chunk], return_tensors="pt",
                                                   truncation=True, max_length=self.TRANSLATE_INPUT_MAX_TOKENS)
                                if self.device >= 0:
                                    inputs = {k: v.cuda()
                                              for k, v in inputs.items()}
                                output = model.generate(**inputs,
                                                        max_new_tokens=self.TRANSLATE_OUTPUT_MAX_TOKENS,
                                                        # PRIMARY FIX: Hard n-gram blocking
                                                        no_repeat_ngram_size=3,  # Prevents any 3-gram from repeating

                                                        # SECONDARY: Soft penalty on repeated tokens
                                                        repetition_penalty=1.2,  # 1.0 = no penalty, 1.5+ can hurt quality

                                                        # Keep deterministic for translation accuracy
                                                        num_beams=4,             # Small beam search improves quality
                                                        do_sample=False,         # Don't sample for translation

                                                        # Performance optimizations
                                                        early_stopping=True,     # Stop when all beams hit EOS
                                                        use_cache=True,
                                                        )
                                translated.append(tokenizer.decode(
                                    output[0],
                                    skip_special_tokens=True))
                                del inputs, output
                            except:
                                translated.append(chunk)
                                torch.cuda.empty_cache()
                    else:
                        raise

        return translated

    def _translate_chunks_nllb(self, chunks: List[str], src_lang: str) -> List[str]:
        """Translate using NLLB (fallback for unsupported languages)."""
        translator = self._load_nllb_translator()
        model = translator['model']
        tokenizer = translator['tokenizer']

        src_code = self.NLLB_LANG_MAP[src_lang]
        tgt_code = self.NLLB_LANG_MAP['en']
        tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_code)
        tokenizer.src_lang = src_code

        print(f"Translating {src_lang} → en (NLLB fallback)")
        print(f"Batch size: 16, Max tokens: 384")  # Conservative for NLLB

        translated = []
        batch_size = 16  # Smaller for NLLB

        with torch.no_grad():
            for i in tqdm(range(0, len(chunks), batch_size), desc="Translating"):
                batch = chunks[i:i + batch_size]

                try:
                    inputs = tokenizer(batch, return_tensors="pt", padding=True,
                                       truncation=True, max_length=384)
                    if self.device >= 0:
                        inputs = {k: v.cuda() for k, v in inputs.items()}

                    outputs = model.generate(
                        **inputs,
                        forced_bos_token_id=tgt_token_id,
                        max_new_tokens=384,
                        num_beams=1,
                        do_sample=False,
                        use_cache=True,
                    )

                    batch_translations = tokenizer.batch_decode(
                        outputs, skip_special_tokens=True)
                    translated.extend(batch_translations)
                    del inputs, outputs

                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                        for chunk in batch:
                            try:
                                inputs = tokenizer([chunk], return_tensors="pt",
                                                   truncation=True, max_length=384)
                                if self.device >= 0:
                                    inputs = {k: v.cuda()
                                              for k, v in inputs.items()}
                                output = model.generate(**inputs, forced_bos_token_id=tgt_token_id,
                                                        max_new_tokens=384, num_beams=1, do_sample=False)
                                translated.append(tokenizer.decode(
                                    output[0], skip_special_tokens=True))
                                del inputs, output
                            except:
                                translated.append(chunk)
                                torch.cuda.empty_cache()
                    else:
                        raise

        return translated

    # ==================== TEXT PROCESSING ====================


    def _detect_word_repetition(self, text: str, threshold: int = 3) -> bool:
        """Enhanced to catch phrase repetitions and character spam."""
        # Original single-word check
        words = text.split()
        if len(words) < 4:
            return False

        # Check for character spam (bullets, dots, etc.)
        char_counts = Counter(text)
        for char, count in char_counts.items():
            if char in '•·*-─' and count > 20:
                return True

        # Check for repeated n-grams (phrases)
        for n in [2, 3, 4]:  # bigrams, trigrams, 4-grams
            if len(words) >= n * threshold:
                ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
                ngram_counts = Counter(ngrams)
                most_common = ngram_counts.most_common(1)
                if most_common and most_common[0][1] > threshold:
                    return True

        # Original consecutive check
        i = 0
        while i < len(words):
            word = words[i]
            count = 1
            while i + count < len(words) and words[i + count] == word:
                count += 1
            if count > threshold:
                return True
            i += count

        return False


    def _is_noise_line(self, line: str) -> bool:
        """Add table/list detection."""
        line = line.strip()
        # ... existing checks ...

        # Detect country/entity lists (likely mangled tables)
        if re.match(r'^[A-Z][a-z]+(\s+[A-Z][a-z]+){10,}$', line):
            return True

        # Detect bullet spam
        if line.count('•') > 5 or line.count('·') > 5:
            return True

        return False

    def _remove_repetitions(self, text: str, max_repeat: int = 2) -> str:
        words = text.split()
        if len(words) < 4:
            return text
        result = []
        i = 0
        while i < len(words):
            word = words[i]
            count = 1
            while i + count < len(words) and words[i + count] == word:
                count += 1
            result.extend([word] * min(count, max_repeat))
            i += count
        return ' '.join(result)

    def _extract_text_pymupdf(self) -> str:
        all_text = []
        try:
            doc = fitz.open(self.pdf_path)
        except Exception as e:
            print(f"  ⚠ PyMuPDF failed to open: {e}")
            return ""

        for page_num, page in enumerate(tqdm(doc, desc="Extracting (PyMuPDF)")):
            try:
                blocks = page.get_text("dict", sort=True)["blocks"]
            except Exception:
                text = page.get_text("text")
                if text:
                    all_text.append(text)
                continue

            page_paragraphs = []
            for block in blocks:
                if block.get("type") != 0:
                    continue
                bbox = block.get("bbox", [0, 0, 0, 0])
                block_height = bbox[3] - bbox[1]
                if block_height < 5:
                    continue

                block_lines = []
                for line in block.get("lines", []):
                    spans_text = []
                    for span in line.get("spans", []):
                        span_text = span.get("text", "").strip()
                        if span_text:
                            spans_text.append(span_text)
                    if spans_text:
                        line_text = " ".join(spans_text)
                        block_lines.append(line_text)

                if block_lines:
                    paragraph = " ".join(block_lines)
                    paragraph = re.sub(r'\s+', ' ', paragraph).strip()
                    if paragraph and len(paragraph) > 2:
                        page_paragraphs.append(paragraph)

            if page_paragraphs:
                all_text.append("\n\n".join(page_paragraphs))

        doc.close()
        return "\n\n".join(all_text)

    def _fix_pdf_encoding(self, text: str) -> str:
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'\(cid:\d+\)', '', text)
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
        for char in ['\u00ad', '\u200b', '\u200c', '\u200d', '\ufeff']:
            text = text.replace(char, '')
        return text

    def _prep_text(self, raw_text: str) -> str:
        text = self._fix_pdf_encoding(raw_text)
        text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
        lines = []
        for line in text.split('\n'):
            line = re.sub(r'[ \t]+', ' ', line).strip()
            if not line:
                lines.append('')
            elif self._is_noise_line(line):
                continue
            else:
                lines.append(line)

        paragraphs = []
        current = []
        for line in lines:
            if line:
                current.append(line)
            elif current:
                para = ' '.join(current)
                if not self._detect_word_repetition(para):
                    para = self._remove_repetitions(para)
                    paragraphs.append(para)
                current = []

        if current:
            para = ' '.join(current)
            if not self._detect_word_repetition(para):
                para = self._remove_repetitions(para)
                paragraphs.append(para)

        return '\n\n'.join(paragraphs)

    def _split_sentences(self, text: str) -> List[str]:
        global _nlp
        doc = _nlp(text)
        return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    def _chunk_text(self, text: str) -> List[str]:
        paragraphs = re.split(r'\n\s*\n', text)
        paragraphs = [p.strip() for p in paragraphs if p.strip()]
        if not paragraphs:
            return []

        chunks = []
        buffer = ""

        for para in paragraphs:
            if self._detect_word_repetition(para):
                continue
            para = self._remove_repetitions(para)
            para_len = len(para)

            if self.MIN_CHARS <= para_len <= self.MAX_CHARS:
                if buffer:
                    combined = buffer + " " + para
                    if len(combined) <= self.MAX_CHARS:
                        chunks.append(combined.strip())
                        buffer = ""
                    else:
                        if len(buffer) >= self.MIN_CHARS:
                            chunks.append(buffer.strip())
                        chunks.append(para)
                        buffer = ""
                else:
                    chunks.append(para)
            elif para_len > self.MAX_CHARS:
                if buffer and len(buffer) >= self.MIN_CHARS:
                    chunks.append(buffer.strip())
                    buffer = ""
                elif buffer:
                    para = buffer + " " + para
                    buffer = ""

                sentences = self._split_sentences(para)
                current = ""
                for sent in sentences:
                    if current and len(current) + len(sent) + 1 > self.MAX_CHARS:
                        if len(current) >= self.MIN_CHARS:
                            chunks.append(current.strip())
                            current = sent
                        else:
                            current = (current + " " + sent).strip()
                            if len(current) >= self.MIN_CHARS:
                                chunks.append(current.strip())
                                current = ""
                    else:
                        current = (current + " " +
                                   sent).strip() if current else sent

                if current:
                    if len(current) >= self.MIN_CHARS:
                        chunks.append(current.strip())
                    else:
                        buffer = current
            else:
                if buffer:
                    combined = buffer + " " + para
                    if len(combined) > self.MAX_CHARS:
                        if len(buffer) >= self.MIN_CHARS:
                            chunks.append(buffer.strip())
                        buffer = para
                    elif len(combined) >= self.MIN_CHARS:
                        chunks.append(combined.strip())
                        buffer = ""
                    else:
                        buffer = combined
                else:
                    buffer = para

        if buffer and len(buffer) >= self.MIN_CHARS:
            chunks.append(buffer.strip())
        elif buffer and chunks:
            if len(chunks[-1]) + len(buffer) + 1 <= self.MAX_CHARS:
                chunks[-1] = chunks[-1] + " " + buffer

        validated_chunks = []
        for chunk in chunks:
            if chunk and not re.search(r'[.!?]\s*$', chunk):
                sentences = self._split_sentences(chunk)
                if len(sentences) > 1:
                    complete = ' '.join(sentences[:-1])
                    if len(complete) >= self.MIN_CHARS:
                        validated_chunks.append(complete.strip())
                    else:
                        validated_chunks.append(chunk.strip())
                else:
                    validated_chunks.append(chunk.strip())
            else:
                validated_chunks.append(chunk.strip())

        return validated_chunks

    def _extract_metadata_from_path(self, pdf_path: Path) -> dict:
        pdf_path = Path(pdf_path)
        filename = pdf_path.stem
        parts = pdf_path.parts
        company = None
        for i, part in enumerate(parts):
            if part.lower() == 'reports' and i + 1 < len(parts):
                company = parts[i + 1]
                break
        if not company:
            skip_folders = {'factsheets', 'highlights',
                            'annual', 'reports', 'data', 'pdfs'}
            for parent in [pdf_path.parent, pdf_path.parent.parent]:
                if parent.name.lower() not in skip_folders and parent.name:
                    company = parent.name
                    break
        company_id = None
        match = re.match(r'^(\d{2,3})_', filename)
        if match:
            company_id = match.group(1)
        year = None
        year_matches = re.findall(r'(20[12]\d)', filename)
        if year_matches:
            year = int(year_matches[0])
        return {'company': company, 'company_id': company_id, 'year': year}

    def set_pdf_path(self, pdf_path: str) -> None:
        new_path = Path(pdf_path)
        if self.pdf_path != new_path:
            self.pdf_path = new_path
            self.prep_data = None
            self.bert_data = None

    def _get_cache_path(self, suffix: str) -> Path:
        if not self.pdf_path:
            raise ValueError("No PDF path set")
        return self.cache_dir / f"{self.pdf_path.stem}_{suffix}.json"

    def _load_prep(self) -> Optional[Dict]:
        if self.prep_data is not None:
            return self.prep_data
        cache_file = self._get_cache_path('prep')
        if cache_file.exists():
            print(f"✓ Loading prep from cache: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.prep_data = json.load(f)
            return self.prep_data
        return None

    def _save_prep(self) -> None:
        if self.prep_data is None:
            return
        cache_file = self._get_cache_path('prep')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.prep_data, f, ensure_ascii=False, indent=2)
        print(f"✓ Cached prep: {cache_file.name}")

    def _load_bert(self) -> Optional[Dict]:
        if self.bert_data is not None:
            return self.bert_data
        cache_file = self._get_cache_path('bert')
        if cache_file.exists():
            print(f"✓ Loading BERT analysis from cache: {cache_file.name}")
            with open(cache_file, 'r', encoding='utf-8') as f:
                self.bert_data = json.load(f)
            return self.bert_data
        return None

    def _save_bert(self) -> None:
        if self.bert_data is None:
            return
        cache_file = self._get_cache_path('bert')
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(self.bert_data, f, ensure_ascii=False, indent=2)
        print(f"✓ Cached BERT analysis: {cache_file.name}")

    def _detect_language(self, text: str) -> str:
        try:
            text_len = len(text)
            samples = []
            if text_len > 15000:
                samples.append(text[1000:4000])
                samples.append(text[text_len//2:text_len//2+3000])
                samples.append(text[-4000:-1000])
            else:
                samples.append(text[100:min(5000, text_len-100)])

            votes = []
            for sample in samples:
                sample_clean = re.sub(r'[^\w\s]', ' ', sample)
                sample_clean = re.sub(r'\s+', ' ', sample_clean)
                if len(sample_clean) > 100:
                    lang, _ = langid.classify(sample_clean)
                    votes.append(lang)

            if votes:
                lang = Counter(votes).most_common(1)[0][0]
                print(f"✓ Language detected: {lang}")
                return lang
            return 'en'
        except Exception as e:
            print(f"⚠ Language detection failed: {e}")
            return 'en'

    # ==================== MAIN PIPELINE ====================

    def extract_pdf(self) -> Dict:
        if not self.pdf_path:
            raise ValueError("No PDF path set")
        if self.pdf_path.is_dir():
            raise ValueError("Cannot extract from directory")

        if self.prep_data is not None:
            return self.prep_data

        cached = self._load_prep()
        if cached:
            return cached

        print(f"\n{'='*80}")
        print(f"STEP 1: EXTRACTING PDF")
        print(f"{'='*80}\n")

        metadata = self._extract_metadata_from_path(self.pdf_path)
        print(
            f"✓ Metadata: company={metadata['company']}, id={metadata['company_id']}, year={metadata['year']}")

        raw_text = self._extract_text_pymupdf()
        print(f"✓ Raw extraction: {len(raw_text):,} characters")

        text = self._prep_text(raw_text)
        print(f"✓ After cleaning: {len(text):,} characters")

        lang = self._detect_language(text)
        chunks = self._chunk_text(text)
        print(f"✓ Created {len(chunks)} chunks")

        if chunks:
            lengths = [len(c) for c in chunks]
            print(
                f"  Chunk sizes: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")

        chunk_ids = []
        for idx in range(len(chunks)):
            if metadata['company_id']:
                chunk_ids.append(f"{metadata['company_id']}_{idx:03d}")
            else:
                chunk_ids.append(f"chunk_{idx:03d}")

        self.prep_data = {
            "pdf_path": str(self.pdf_path),
            "company": metadata['company'],
            "company_id": metadata['company_id'],
            "year": metadata['year'],
            "language": lang,
            "extraction_method": "pymupdf_blocks",
            "num_chunks": len(chunks),
            "chunks": chunks,
            "chunk_ids": chunk_ids,
            "translated": False,
            "extracted_at": str(datetime.now())
        }

        self._save_prep()
        return self.prep_data

    def translate_pdf(self) -> Dict:
        if not self.pdf_path:
            raise ValueError("No PDF path set")

        if self.prep_data and self.prep_data.get('translated'):
            return self.prep_data

        if not self.prep_data:
            cached = self._load_prep()
            if cached and cached.get('translated'):
                return cached

        print(f"\n{'='*80}")
        print(f"STEP 2: TRANSLATING")
        print(f"{'='*80}\n")

        extracted = self.extract_pdf()
        lang = extracted['language']
        chunks = extracted['chunks']

        if lang == 'en':
            print(f"✓ Already in English")
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['translated_at'] = str(datetime.now())
        elif self.use_helsinki and lang in self.HELSINKI_MODELS:
            # Use fast Helsinki-NLP model
            start = time.time()
            translated_chunks = self._translate_chunks_helsinki(chunks, lang)
            elapsed = time.time() - start
            chunks_per_sec = len(translated_chunks) / elapsed
            print(
                f"✓ Translated {len(translated_chunks)} chunks in {elapsed:.1f}s ({chunks_per_sec:.2f} chunks/sec)")

            self.prep_data['chunks'] = translated_chunks
            self.prep_data['chunk_pairs'] = [
                {"original": o, "translated": t} for o, t in zip(chunks, translated_chunks)
            ]
            self.prep_data['translated'] = True
            self.prep_data['translation_model'] = 'helsinki'
            self.prep_data['translated_at'] = str(datetime.now())
            self._unload_translator()

        elif lang in self.NLLB_LANG_MAP:
            # Fallback to NLLB
            start = time.time()
            translated_chunks = self._translate_chunks_nllb(chunks, lang)
            elapsed = time.time() - start
            chunks_per_sec = len(translated_chunks) / elapsed
            print(
                f"✓ Translated {len(translated_chunks)} chunks in {elapsed:.1f}s ({chunks_per_sec:.2f} chunks/sec)")

            self.prep_data['chunks'] = translated_chunks
            self.prep_data['chunk_pairs'] = [
                {"original": o, "translated": t} for o, t in zip(chunks, translated_chunks)
            ]
            self.prep_data['translated'] = True
            self.prep_data['translation_model'] = 'nllb'
            self.prep_data['translated_at'] = str(datetime.now())
            self._unload_translator()
        else:
            print(f"⚠ Translation not supported for: {lang}")
            self.prep_data['chunk_pairs'] = [
                {"original": c, "translated": c} for c in chunks]
            self.prep_data['unsupported'] = True
            self.prep_data['translated_at'] = str(datetime.now())

        self._save_prep()
        return self.prep_data

    # ==================== BERT ANALYSIS ====================

    def load_model(self, model_name: str, task_name: str):
        if task_name in self.models:
            return self.models[task_name]

        if self.models:
            self._clear_gpu_memory()

        print(f"Loading {task_name} model: {model_name}")
        self.models[task_name] = pipeline(
            "text-classification",
            model=model_name,
            device=self.device,
            batch_size=self.BATCH_SIZE
        )
        return self.models[task_name]

    def filter_climate_chunks(self) -> Dict:
        if self.bert_data and self.bert_data.get('filtered'):
            return self.bert_data

        cached = self._load_bert()
        if cached and cached.get('filtered'):
            return cached

        if not self.prep_data:
            self.prep_data = self._load_prep()
            if not self.prep_data:
                raise FileNotFoundError("Run extract_pdf() first")

        if self.prep_data.get('translated'):
            chunks = self.prep_data['chunks']
            source = "translated"
        else:
            chunks = self.prep_data['chunks']
            source = "original"

        chunk_ids = self.prep_data.get(
            'chunk_ids', [f"chunk_{i:03d}" for i in range(len(chunks))])

        print("\nPreparing for BERT analysis...")
        self._unload_translator()
        self._clear_gpu_memory()

        if self.device >= 0:
            mem_allocated = torch.cuda.memory_allocated(0) / 1e9
            print(f"  GPU memory: {mem_allocated:.2f}GB allocated")

        detector = self.load_model(
            "climatebert/distilroberta-base-climate-detector", "detector")

        print(
            f"\nFiltering {len(chunks)} {source} chunks for climate content...")
        climate_chunks = []

        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Detecting"):
            batch = chunks[i:i+self.BATCH_SIZE]
            batch_ids = chunk_ids[i:i+self.BATCH_SIZE]
            try:
                results = detector(batch, truncation=True, max_length=512)
                for chunk, chunk_id, result in zip(batch, batch_ids, results):
                    if result['label'].lower() in ['climate', 'yes', 'climate-related']:
                        climate_chunks.append({
                            'chunk_id': chunk_id,
                            'company': self.prep_data.get('company'),
                            'company_id': self.prep_data.get('company_id'),
                            'year': self.prep_data.get('year'),
                            'text': chunk,
                            'detector_score': result['score'],
                            'detector_label': result['label']
                        })
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        kept_pct = len(climate_chunks)/len(chunks)*100 if chunks else 0
        print(f"✓ Kept {len(climate_chunks)} climate chunks ({kept_pct:.1f}%)")

        self.bert_data = {
            'pdf_path': str(self.pdf_path),
            'company': self.prep_data.get('company'),
            'company_id': self.prep_data.get('company_id'),
            'year': self.prep_data.get('year'),
            'language': self.prep_data.get('language', 'unknown'),
            'source': source,
            'filtered': True,
            'filtered_at': datetime.now().isoformat(),
            'total_chunks': len(chunks),
            'climate_chunks': len(climate_chunks),
            'kept_percentage': kept_pct,
            'chunks': climate_chunks
        }

        self._save_bert()
        return self.bert_data

    def analyze_specificity(self) -> Dict:
        if self.bert_data and self.bert_data.get('specificity_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            self.filter_climate_chunks()

        chunks = self.bert_data['chunks']

        if 'detector' in self.models:
            del self.models['detector']
            self._clear_gpu_memory()

        specificity = self.load_model(
            "climatebert/distilroberta-base-climate-specificity", "specificity")

        print(f"\nAnalyzing specificity for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Specificity"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = specificity(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['specificity_label'] = result['label']
                    chunk['specificity_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['specificity_analyzed'] = True
        self.bert_data['specificity_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_sentiment(self) -> Dict:
        if self.bert_data and self.bert_data.get('sentiment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('specificity_analyzed'):
            self.analyze_specificity()

        chunks = self.bert_data['chunks']

        if 'specificity' in self.models:
            del self.models['specificity']
            self._clear_gpu_memory()

        sentiment = self.load_model(
            "climatebert/distilroberta-base-climate-sentiment", "sentiment")

        print(f"\nAnalyzing sentiment for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Sentiment"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = sentiment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['sentiment_label'] = result['label']
                    chunk['sentiment_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['sentiment_analyzed'] = True
        self.bert_data['sentiment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def analyze_commitments(self) -> Dict:
        if self.bert_data and self.bert_data.get('commitment_analyzed'):
            return self.bert_data
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data or not self.bert_data.get('sentiment_analyzed'):
            self.analyze_sentiment()

        chunks = self.bert_data['chunks']

        if 'sentiment' in self.models:
            del self.models['sentiment']
            self._clear_gpu_memory()

        commitment = self.load_model(
            "climatebert/distilroberta-base-climate-commitment", "commitment")

        print(f"\nAnalyzing commitments for {len(chunks)} chunks...")
        for i in tqdm(range(0, len(chunks), self.BATCH_SIZE), desc="Commitments"):
            batch_chunks = chunks[i:i+self.BATCH_SIZE]
            batch_texts = [c['text'] for c in batch_chunks]
            try:
                results = commitment(
                    batch_texts, truncation=True, max_length=512)
                for chunk, result in zip(batch_chunks, results):
                    chunk['commitment_label'] = result['label']
                    chunk['commitment_score'] = result['score']
            except Exception as e:
                print(f"⚠ Error: {e}")
                self._clear_gpu_memory()

        self.bert_data['commitment_analyzed'] = True
        self.bert_data['commitment_analyzed_at'] = datetime.now().isoformat()
        self._save_bert()
        return self.bert_data

    def process_pdfs(self, path: str, extract: bool = True, translate: bool = True,
                     filter_climate: bool = True, analyze: bool = True,
                     skip_errors: bool = True) -> Dict:
        target = Path(path)
        if not target.exists():
            print(f"❌ Path not found: {path}")
            return None

        if target.is_file():
            if target.suffix.lower() != '.pdf':
                print(f"❌ Not a PDF file: {path}")
                return None
            pdf_files = [target]
            is_single = True
            root = target.parent
        else:
            pdf_files = sorted(target.rglob("*.pdf"))
            if not pdf_files:
                print(f"❌ No PDFs found in {path}")
                return None
            is_single = False
            root = target

        print(f"\n{'='*80}")
        print(f"{'PROCESSING' if is_single else 'BATCH PROCESSING'}")
        print(f"Found {len(pdf_files)} PDF(s)")
        print(
            f"Translation: {'Helsinki-NLP (fast)' if self.use_helsinki else 'NLLB (universal)'}")
        print(f"{'='*80}\n")

        stats = {
            'total': len(pdf_files), 'extracted': 0, 'translated': 0,
            'skipped_translation': 0, 'filtered': 0, 'analyzed': 0, 'errors': [],
            'start_time': time.time()
        }

        for i, pdf_file in enumerate(pdf_files, 1):
            relative_path = pdf_file.relative_to(root)
            print(f"\n[{i}/{len(pdf_files)}] {relative_path}")

            try:
                self.set_pdf_path(str(pdf_file))

                if extract:
                    self.extract_pdf()
                    stats['extracted'] += 1

                if translate:
                    result = self.translate_pdf()
                    if result.get('translated'):
                        stats['translated'] += 1
                    else:
                        stats['skipped_translation'] += 1

                if filter_climate:
                    filtered = self.filter_climate_chunks()
                    stats['filtered'] += 1
                    print(
                        f"  ✓ {filtered['climate_chunks']}/{filtered['total_chunks']} climate chunks")

                if analyze and filter_climate:
                    self.analyze_specificity()
                    self.analyze_sentiment()
                    self.analyze_commitments()
                    stats['analyzed'] += 1

                self._clear_gpu_memory()

            except Exception as e:
                stats['errors'].append(
                    {'file': str(relative_path), 'error': str(e)})
                if not skip_errors:
                    raise
                print(f"  ⚠ Error: {e}")
                self._emergency_gpu_cleanup()

        stats['elapsed_time'] = time.time() - stats['start_time']
        stats['avg_time_per_pdf'] = stats['elapsed_time'] / \
            len(pdf_files) if pdf_files else 0

        print(f"\n{'='*80}")
        print(f"COMPLETE")
        print(f"  Extracted: {stats['extracted']}")
        print(f"  Translated: {stats['translated']}")
        print(f"  Filtered: {stats['filtered']}")
        print(f"  Analyzed: {stats['analyzed']}")
        print(f"  Errors: {len(stats['errors'])}")
        print(f"  Total time: {stats['elapsed_time']/60:.1f} min")
        print(f"  Avg per PDF: {stats['avg_time_per_pdf']:.1f}s")
        print(f"{'='*80}\n")

        return stats

    def inspect_extraction(self, n_samples=3):
        if not self.prep_data:
            self.prep_data = self._load_prep()
        if not self.prep_data:
            print("No data. Run extract_pdf() first.")
            return

        chunks = self.prep_data['chunks']
        lengths = [len(c) for c in chunks]

        print(f"\n{'='*80}")
        print(
            f"EXTRACTION: {self.pdf_path.name if self.pdf_path else 'unknown'}")
        print(f"Company: {self.prep_data.get('company')}")
        print(f"Year: {self.prep_data.get('year')}")
        print(f"Language: {self.prep_data['language']}")
        print(f"Method: {self.prep_data.get('extraction_method', 'unknown')}")
        print(f"Total chunks: {len(chunks)}")
        print(f"Avg size: {sum(lengths)/len(lengths):.0f} chars")
        print(f"Min/Max: {min(lengths)} / {max(lengths)} chars")
        print(f"{'='*80}\n")

        import random
        samples = random.sample(chunks, min(n_samples, len(chunks)))
        for i, chunk in enumerate(samples, 1):
            print(f"{'─'*80}")
            print(f"Sample {i} ({len(chunk)} chars):")
            print(f"{'─'*80}")
            print(chunk[:500] + ('...' if len(chunk) > 500 else ''))
            print()

    def get_final_results(self) -> Optional[Dict]:
        if not self.bert_data:
            self.bert_data = self._load_bert()
        if not self.bert_data:
            print(f"❌ No analysis found")
            return None

        required = ['filtered', 'specificity_analyzed',
                    'sentiment_analyzed', 'commitment_analyzed']
        missing = [r for r in required if not self.bert_data.get(r)]
        if missing:
            print(f"⚠ Incomplete analysis. Missing: {', '.join(missing)}")

        return self.bert_data